# M1 linear modes and dynamics

This offline notebook runs the package CLI on the committed tiny fixture. The resulting modes are slow-mode candidates, not proof of order parameters.

In [ ]:
from pathlib import Path
import json
import tempfile

from semmap_haken.cli import main
from semmap_haken.compute import ComputeContext
from semmap_haken.manifest import RunManifest
from semmap_haken.notebook import resolve_notebook_paths, run_resource_preflight

ALLOW_PRODUCTION_DOWNLOAD = False
EXECUTION = {'backend': 'auto', 'device': 0, 'dtype': 'float64', 'workers': 'auto', 'reserved_cpu_cores': 1, 'threads_per_worker': 1, 'gpu_memory_fraction': 0.80, 'batch_size': 'auto', 'deterministic': True, 'allow_auto_fallback': True}
accelerator = ComputeContext.create(**EXECUTION)
print({'selected_execution': accelerator.telemetry(), 'note': 'CUDA parity is optional; do not interpret timing as a speedup claim without a recorded comparison.'})
fixture = Path('tests/fixtures/conceptnet_tiny.tsv').resolve()
assert fixture.is_file(), 'Run from the repository root or supply a prepared artifact manually.'
work = Path(tempfile.mkdtemp(prefix='semmap-haken-m1-'))
config = work / 'experiment.yaml'
config.write_text(f'''
paths: {{workspace_root: {work}, data_root: data, cache_root: cache, runs_root: runs}}
dataset: {{source: fixture, path: {fixture}, language: en, relations: [], min_weight: 1.0, max_nodes: 10, component: all}}
graph: {{directed: false, weight_transform: raw, operator: normalized_adjacency}}
runtime: {{profile: smoke, random_seed: 1729}}
execution: {json.dumps(EXECUTION)}
dynamics: {{model: linear, alpha: 1.0, beta: 0.5, time_start: 0.0, time_stop: 2.0, time_steps: 11, perturbations_per_kind: 1, storage_policy: all}}
spectral: {{top_k: 3, max_r: 2}}
''', encoding='utf-8')
paths = resolve_notebook_paths(config)
preflight = run_resource_preflight(None, paths.runs_root)
print({'offline': not ALLOW_PRODUCTION_DOWNLOAD, 'preflight_ok': preflight.ok, 'workspace': str(work)})

In [ ]:
# Lightweight benchmark control: repeat only after changing one execution setting.
# The run artifacts record timings, selected batch size, and fallback reasons.
import time
benchmark_started = time.perf_counter()
print({'benchmark_control': EXECUTION, 'started_at_seconds': benchmark_started})

In [ ]:
assert main(['prepare', '--config', str(config)]) == 0
prepared = sorted((work / 'runs').glob('prepare-*'))[-1]
text = config.read_text(encoding='utf-8').replace('spectral: {top_k: 3, max_r: 2}', f'spectral: {{prepared_graph_dir: {prepared}, top_k: 3, max_r: 2}}')
config.write_text(text, encoding='utf-8')
assert main(['run', '--config', str(config)]) == 0
run_dir = sorted((work / 'runs').glob('run-*'))[-1]
manifest = RunManifest.read_json(run_dir / 'manifest.json')
summary = json.loads((run_dir / 'dynamics' / 'dynamics_summary.json').read_text(encoding='utf-8'))
spectral = json.loads((run_dir / 'spectral' / 'spectral_diagnostics.json').read_text(encoding='utf-8'))
print({'run_id': manifest.run_id, 'paths': manifest.artifacts, 'beta_rule': spectral['beta_selection'], 'r_candidate': spectral['selection']['selected_r'], 'trajectory_error': summary['aggregate']['mean_relative_rmse']})